**Versão adaptada para corpus dividido** (corpus => treinamento, validacao e teste)

Tutorial: https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html

In [ ]:
# Imports
import time
import math
from __future__ import unicode_literals, print_function, division
from io import open

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# BLEU Score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

O torch é a base para manipulação de tensores (estruturas especializadas em armazenar dados numéricos). O módulo nn fornece os blocos construtores para o Encoder e o Decoder (como camadas de Embedding), enquanto o F (functional) oferece funções como ativações e log-softmax

In [ ]:
# Tokens especiais
SOS_token = 0 # Start of Sequence
EOS_token = 1 # End of Sequence
PAD_token = 2 # Padding
UNK_token = 3 # Unknown

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {
            SOS_token: "SOS", 
            EOS_token: "EOS",
            PAD_token: "PAD",
            UNK_token: "UNK"
        }
        self.n_words = 4  # Contando os tokens especiais

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

A classe Lang é responsável por construir o vocabulário da linguagem, realizando o mapeamento de cada palavra e seus respectivos índices, além da frequência de palavras.

Os métodos addSentence e addWord são responsáveis por atualizar automaticamente o vocabulário, a partir das sentenças do dataset.

Porém, não podemos permitir que qualquer tipo de termo possa ser adicionado ao vocabulário. As palavras devem passar por uma padronização, que será realizada em corpus.ipynb (responsável pela divisão do corpus).


Isso contribuirá para a redução de ruído no dataset e irá melhorar a consistência dos tokens utilizados pelo modelo.

Agora, com a etapa de tratamento dos dados finalizada, o próximo passo é importar o dataset linguístico que será utilizado para terinar o modelo

In [ ]:
def readCorpus(filename):
    print(f"Lendo: {filename}")

    with open(filename, encoding="utf-8") as file:
        # Lê o dataset e o divide em linhas
        lines = file.read().strip().split('\n')

    pairs = []

    for line in lines:
        parts = line.split('\t')

        if len(parts) != 2:
            continue

        source = parts[0]
        target = parts[1]
        
        pairs.append([source, target])

    print(f"Total: {len(pairs)} pares de sentenças lidas")

    return pairs

In [ ]:
# Lendo os 3 corpora zo'é -> protuguês
train_pairs = readCorpus("treinamento.txt")
valid_pairs = readCorpus("validacao.txt")
test_pairs = readCorpus("teste.txt")

In [ ]:
# Criando vocabulário
input_lang = Lang("zoe")
output_lang = Lang("por")

for pair in train_pairs:
    input_lang.addSentence(pair[0])
    output_lang.addSentence(pair[1])

print("Vocabulários construídos a partir do corpus de treinamento:")
print(f"zoe: {input_lang.n_words} palavras")
print(f"por: {output_lang.n_words} palavras")

Como trata-se de um exemplo, vamos aplicar filtros no dataset para limitar o tamanho das sequências e restringir as estruturas sintáticas das frases. Assim, com a redução da complexidade do problema, o treinamento do modelo será mais rápido neste ambiente de demonstração.

In [ ]:
# Já realizado em corpus.ipynb

MAX_LENGTH = 10

# def filterPair(p):
#     return len(p[0].split(' ')) < MAX_LENGTH and \
#         len(p[1].split(' ')) < MAX_LENGTH


# def filterPairs(pairs):
#     return [pair for pair in pairs if filterPair(pair)]

Agora, vamos preparar os dados. Esse processo se baseia nas seguintes etapas:
*  Ler dataset, dividí-lo em linhas e, após isso, dividir as linhas em pares.
*  Normalizar o texto e aplicar os filtros
*  Criar listas de palavras a partir de frases em duplas

Ou seja, vamos aplicar tudo o que foi construído até agora e montar o vocabulário final que será utilizado pelo modelo.

É aqui que vamos iniciar o processo de treinamento modelo Seq2Seq.<br>
Mas, primeiramente, precisamos preparar os dados de trinamento: para cada par, precisaremos de um tensor de entrada (índices das palavras na frase de entrada) e um tensor alvo (índices das palavras na frase alvo). Ao criar esses vetores, adicionaremos o token EOS a ambas as sequências.

(Algumas validações extras antes de continuarmos...)

In [ ]:
maior = 0
maior_sentenca = ""
qtd = 0

for pair in train_pairs:
    quantidade = len(pair[0].split())

    if quantidade > 10:
        qtd += 1

    if quantidade > maior:
        maior = quantidade
        maior_sentenca = pair[0]

print("Maior quantidade de palavras no source:", maior)
print("Sentença:", maior_sentenca)
print(qtd)

In [ ]:
maior = 0
maior_sentenca = ""
qtd2 = 0

for pair in train_pairs:
    quantidade = len(pair[1].split())

    if quantidade > 10:
        qtd2 += 1

    if quantidade > maior:
        maior = quantidade
        maior_sentenca = pair[1]

print("Maior quantidade de palavras no target:", maior)
print("Sentença:", maior_sentenca)
print(qtd2)

In [ ]:
print(len(train_pairs))
print(len(valid_pairs))
print(len(test_pairs))

In [ ]:
# Converte palavras em índices usando o vocabulário construído
# A função dos índices obtidos é representar cada palavra como um identificador numérico único no vocabulário
def indexesFromSentence(lang, sentence):
    return [
        # Retorna a palavra, caso seja encontrada no vocabulário. Caso contrário, retorna UNK_token
        lang.word2index.get(word, UNK_token)
        for word in sentence.split(' ')
    ]

# Transforma a lista gerada em indexesFromSentence() em um tensor PyTorch
def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

# Junta input e output
def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(pairs, batch_size, shuffle=False):
    n = len(pairs)

    # Inicializando os vetores com PAD_token
    # (n, MAX_LENGTH) = shape do vetor => n linhas e MAX_LENGTH+1 colunas
        # MAX_LENGTH+1: considerando um espaço a mais para EOS token
    input_ids = np.full((n, MAX_LENGTH+1), PAD_token, dtype=np.int32)
    target_ids = np.full((n, MAX_LENGTH+1), PAD_token, dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    dataset = TensorDataset(
        torch.LongTensor(input_ids).to(device),
        torch.LongTensor(target_ids).to(device)
    )

    # Embaralha a ordem em que sentenças aparecem para criar dataset (de treinamento)
    if shuffle:
        # Seleciona uma amostragem aleatória
        sampler = RandomSampler(dataset)
        dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)
    else:
        dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader

A função *get_dataloader* é responsável por preparar todos os dados para o treinamento, convertendo os pares de frases em tensores e criando um DataLoader.<br>
Funções das classes importadas:
*  TensorDataset: empacota múltiplos tensores em um único dataset PyTorch, onde cada amostra é uma tupla contendo um elemento de cada tensor
*  RandomSampler: seleciona uma amostragem aleatória dos dataset de dados de treinamento
*  DataLoader: gerencia como as amostras do dataset são entregues durante o treinamento

___

Agora, avançaremos para a primeira parte de Deep Learning: o encoder

A classe *EncoderRNN* é responsável por criar uma rede neural PyTorch.<br>
nn.Module fornece:
*  gerenciamento de parâmetros
*  integração com autograd
*  estrutura para treinamento
<br>

Diferente de uma RNN simples, a GRU usa "portões" (gates) internos — o update gate (portão de atualização) e o reset gate (portão de redefinição). Esses portões decidem quais informações devem ser passadas adiante e quais devem ser esquecidas, otimizando o fluxo de dados.

In [ ]:
class EncoderRNN(nn.Module):
    # input_size = número de palavras do vocabulário de entrada
    # hidden_size = tamanho do vetor interno da RNN (ex: 256, 512)
        # define a dimensão da representação aprendida
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        # Transforma o índice da palavra em um vetor de dimensão hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)

        # GRU = Gated Recurrent Unit (Unidade Recorrente com Portões)
            # é um tipo de RNN
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        # Desliga aleatoriamente uma porcentagem dos neurônios durante o treinamento
            # impede que o modelo decore padrões específicos dos dados de treino
        self.dropout = nn.Dropout(dropout_p)
    
    # Define como os dados passam pela rede
    def forward(self, input):
        # Cria o embedding com base no índice da palavra
        embedded = self.dropout(self.embedding(input))
        
        # Output = todos os estados ocultos
        # Hidden = último estado oculto
        output, hidden = self.gru(embedded)
        return output, hidden

Agora, chegamos na outra metade fundamental da arquitetura Seq2Seq: o decoder.<br>
Em cada etapa da decodificação, o decoder recebe um token de entrada e um estado oculto (hidden).<br>
O token de entrada inicial é o SOS (token de início da sequência) e o primeiro estado oculto é o vetor de contexto (o último estado oculto do encoder)

In [ ]:
class DecoderRNN(nn.Module):
    # hidden_size = tamanho do vetor interno da RNN (mesmo do encoder)
    # output_size = número de palavras do vocabulário de saída
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        # Camada que gera uma pontuação para cada token com base no vocabulário
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0) # Tamanho do batch
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden # Vetor de contexto
        decoder_outputs = [] # Lista para armazenar todas as previsões

        for i in range(MAX_LENGTH+1):
            # Determina previsão para a próxima palavra
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            # Modo de treinamento
            if target_tensor is not None:
                # Teacher forcing: usa a palavra CORRETA como próximo input

                # Seleciona todos os elementos de dimensão 0 que estão na posição i da sequência
                decoder_input = target_tensor[:, i].unsqueeze(1)
            
            # Modo de inferência
            else:
                # A própria previsão do modelo é usada como próximo input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()   # remove a última dimensão
                                                                # ex: tensor([[[2], [2]]]) -> tensor([[2], [2]])
        
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        
        # decoder_outputs = probabilidades das palavras geradas
        # decoder_hidden = último estado oculto
        # None = placeholder para outros mecanismos complementares (ex: atenção)
        return decoder_outputs, decoder_hidden, None

    # Processa um único passo da decodificação, transformando uma palavra de entrada e um estado oculto em:
        # uma previsão para a próxima palavra (output)
        # um novo estado oculto atualizado (hidden)
    def forward_step(self, input, hidden):
        output = self.embedding(input)
        # Introduz não-linearidade
            # elimina valores negativos (são zerados)
        output = F.relu(output)
        # Processamento da RNN
            # palavra atual + estado anterior -> novo estado
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        
        # Output = logits (pontuações brutas) para a próxima palavra
        # Hidden = novo estado oculto
        return output, hidden

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH+1):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

Vamos configurar nossa época de treinamento. Isto é, uma passagem completa de todo o conjunto de dados de treinamento pelo modelo

In [ ]:
# Executa uma época completa de treinamento
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        # Limpa os gradientes do passo anterior
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )

        # Calcula gradientes
        loss.backward()

        # Atualiza os pesos
        encoder_optimizer.step()
        decoder_optimizer.step()

        # Soma de loss
        total_loss += loss.item()

    return total_loss / len(dataloader) # Loss média da época

In [ ]:
# Época de validação
    # similar ao treinamento, porém sem atualização de gradientes e demais parâmetros
    # medirá o desempenho, sem modificar os pesos 

def validate_epoch(dataloader, encoder, decoder, criterion):
    total_loss = 0

    encoder.eval()
    decoder.eval()

    # desativa o cálculo dos gradientes
    with torch.no_grad():
        for data in dataloader:
            input_tensor, target_tensor = data

            encoder_outputs, encoder_hidden = encoder(input_tensor)
            decoder_outputs, _, _ = decoder(encoder_outputs,encoder_hidden,target_tensor)

            loss = criterion(
                decoder_outputs.view(-1, decoder_outputs.size(-1)),
                target_tensor.view(-1)
            )

            total_loss += loss.item()

    encoder.train()
    decoder.train()

    return total_loss / len(dataloader)

Vamos criar uma função auxiliar para acompanharmos o tempo decorrido e o tempo restante estimado, dado o tempo atual e o progresso em porcentagem.

In [ ]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

Processo de treinamento:
*  Inicia um cronômetro
*  Inicializa os otimizadores e critério
*  Cria conjunto de pares de treinamento

Após isso, faremos várias chamadas da função *train*, imprimindo o progresso e a perda média

*  optim.Adam: é um otimizador. Algoritmo responsável por atualizar os pesos da rede durante o treinamento.
*  encoder.parameters(): parâmetros do modelo. Responsável por retornar todos os pesos treináveis do encoder (embeddings, GRU, etc)
*  nn.NNLLoss: função de perda. Responsável por medir o erro da previsão, trabalhando com probabilidades em forma de log

Vamos implementar a avaliação do modelo.<br>
É similar ao processo de treinamento, porém como não há alvos vamos realimentar as previsões do decoder para ele mesmo a cada passo.

In [ ]:
# Testa o modelo traduzindo uma única frase
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    # desativa o cálculo de gradientes, pois só queremos realizar uma previsão
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        # Pega os índices das palavras mais prováveis
        _, topi = decoder_outputs.topk(1) # Pega o índice de maior probabilidade
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            # Converte índice para palavra
            decoded_words.append(output_lang.index2word[idx.item()])

    return decoded_words, decoder_attn

In [ ]:
# Avalia o modelo por meio do corpus de teste
def evaluateTestSet(encoder, decoder, test_pairs):
    results = []

    encoder.eval()
    decoder.eval()

    for source, target in test_pairs:
        output_words, _ = evaluate(encoder, decoder,  source, input_lang, output_lang)

        # Remove <EOS>
        temp = []
        for word in output_words:
            if word != '<EOS>':
               temp.append(word)
              
        output_words = temp

        prediction = ' '.join(output_words)

        results.append({
            "source": source,
            "target": target,
            "prediction": prediction
        })

    return results

In [ ]:
# Calcula BLEU Score em um conjunto de pares de validação
def calculate_validate_bleu(encoder, decoder, pairs, input_lang, output_lang):
    encoder.eval()
    decoder.eval()

    references = []
    predictions = []

    with torch.no_grad():
        for source, target in pairs:
            output_words, _ = evaluate(encoder, decoder, source, input_lang, output_lang)

            # Remove <EOS>
            tmp = []
            for word in output_words:
                if word != '<EOS>':
                    tmp.append(word)
                    
            output_words = tmp

            references.append([target.split()])
            predictions.append(output_words)
    
    encoder.train()
    decoder.train()

    # evita que a bleu final seja zerada caso algum n-gram apresente bleu=0
    smoothie = SmoothingFunction().method4
    
    return corpus_bleu(references, predictions, smoothing_function=smoothie)

In [ ]:
def train(train_dataloader, valid_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, valid_pairs_bleu=None, bleu_every=10):

    start = time.time()

    train_losses = []
    valid_losses = []

    print_train_loss_total = 0 # Reseta a cada print_every
    print_valid_loss_total = 0 # Reseta a cada print_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=PAD_token)

    # Registra melhor desempenho por loss de validação
    best_valid_loss = float('inf')
    best_valid_loss_epoch = 0

    # Registra melhor desempenho por pontuação BLEU
    best_bleu = 0.0
    best_bleu_epoch = 0

    for epoch in range(1, n_epochs + 1):
        # Treinamento
        # Treina uma época completa
        train_loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)

        # Validação
        valid_loss = validate_epoch(valid_dataloader, encoder, decoder, criterion)

        # Acumula as perdas
        print_train_loss_total += train_loss
        print_valid_loss_total += valid_loss
        
        # Armazena as perdas
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        # Salva melhor valid_loss
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_loss_epoch = epoch

            torch.save(encoder.state_dict(), "best_valid_loss_encoder.pt")
            torch.save(decoder.state_dict(), "best_valid_loss_decoder.pt")

        # Salva melhor modelo em sua melhor pontuação BLEU
        if valid_pairs_bleu is not None and epoch % bleu_every == 0:
            current_bleu = calculate_validate_bleu(encoder, decoder, valid_pairs_bleu, input_lang, output_lang)

            if current_bleu > best_bleu:
                best_bleu = current_bleu
                best_bleu_epoch = epoch

                torch.save(encoder.state_dict(), "best_bleu_encoder.pt")
                torch.save(decoder.state_dict(), "best_bleu_decoder.pt")

        # Imprimindo dados
        if epoch % print_every == 0:
            print_train_loss_avg = print_train_loss_total / print_every
            print_train_loss_total = 0

            print_valid_loss_avg = print_valid_loss_total / print_every
            print_valid_loss_total = 0

            print('%s (%d %d%%) train_loss: %.4f | valid_loss: %.4f' % (
                timeSince(start, epoch / n_epochs), # tempo decorrido
                epoch, # época atual
                epoch / n_epochs * 100, # porcentagem concluída
                print_train_loss_avg, # perda média do treino
                print_valid_loss_avg # perda média da validação
            ))

    print(f"\nMelhor valid_loss: {best_valid_loss:.4f} na época {best_valid_loss_epoch}")

    if valid_pairs_bleu is not None:
        print(f"Melhor BLEU Score: {best_bleu:.4f} na época {best_bleu_epoch}")

    return train_losses, valid_losses, best_valid_loss_epoch, best_bleu_epoch

In [ ]:
# BLEU Score
def calculate_final_bleu_score(test_results):
    references = []
    predictions = []

    for res in test_results:
        target_tokens = res["target"].split()
        prediction_tokens = res["prediction"].split()

        references.append([target_tokens])
        predictions.append(prediction_tokens)

    # evita que a bleu final seja zerada caso algum n-gram apresente bleu=0
    smoothie = SmoothingFunction().method4

    bleu1 = corpus_bleu(references, predictions, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu2 = corpus_bleu(references, predictions, weights=(0, 1, 0, 0), smoothing_function=smoothie)
    bleu3 = corpus_bleu(references, predictions, weights=(0, 0, 1, 0), smoothing_function=smoothie)
    bleu4 = corpus_bleu(references, predictions, weights=(0, 0, 0, 1), smoothing_function=smoothie)

    final_bleu = corpus_bleu(references, predictions, smoothing_function=smoothie)

    print(f"1-gram Score: {bleu1:.4f}")
    print(f"2-gram Score: {bleu2:.4f}")
    print(f"3-gram Score: {bleu3:.4f}")
    print(f"4-gram Score: {bleu4:.4f}")
    print(f"BLEU Score: {final_bleu:.4f}\n")

In [ ]:
def evaluateTranslation(encoder_eval, decoder_eval, best_encoder_path, best_decoder_path, pairs):
    # Carrega encoder
    encoder_eval.load_state_dict(torch.load(best_encoder_path))

    # Carrega decoder
    decoder_eval.load_state_dict(torch.load(best_decoder_path))

    test_results = evaluateTestSet(encoder_eval, decoder_eval, pairs)
    print(f"Quantidade de traduções avaliadas: {len(test_results)}\n")

    # Calcula BLEU Score do conjunto de traduções em test_results
    calculate_final_bleu_score(test_results)

    return test_results

In [ ]:
# Visualizando traduções geradas
def showTranslations(translations, qtd):
    for result in translations[:qtd]:
        print("Zo'é:       ", result["source"])
        print("Referência: ", result["target"])
        print("Predição:   ", result["prediction"])
        print()

In [ ]:
def count_matches(test_results):
    matches = []

    for res in test_results:
        target_tokens = res["target"].split()
        prediction_tokens = res["prediction"].split()

        if prediction_tokens == target_tokens:
            matches.append(res)

    total = len(test_results)
    qtd_matches = len(matches)
    pct = (qtd_matches / total * 100) if total > 0 else 0.0

    print(f"Traduções idênticas à referência: {qtd_matches}/{total} ({pct:.2f}%)")

    return matches

**Programa principal**

In [ ]:
# TESTE - Programa principal
hidden_size = 128
batch_size = 32
n_epochs = 300

train_dataloader = get_dataloader(train_pairs, batch_size, shuffle=True)
valid_dataloader = get_dataloader(valid_pairs, batch_size)
test_dataloader = get_dataloader(test_pairs, batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train_losses, valid_losses, best_valid_loss_epoch, best_bleu_epoch = train(train_dataloader, valid_dataloader, encoder, decoder, n_epochs, print_every=5, valid_pairs_bleu=valid_pairs, bleu_every=1)

In [ ]:
# Testando o modelo configurado a partir do treinamento com melhor desempenho baseado na loss de validação
valid_encoder_eval = EncoderRNN(input_lang.n_words, hidden_size).to(device)
#valid_decoder_eval = DecoderRNN(hidden_size, output_lang.n_words).to(device)
valid_decoder_eval = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

best_valid_translations = evaluateTranslation(valid_encoder_eval, valid_decoder_eval, "best_valid_loss_encoder.pt", "best_valid_loss_decoder.pt", test_pairs)

In [ ]:
# Testando o modelo configurado a partir do treinamento com melhor desempenho baseado na BLEU Score
bleu_encoder_eval = EncoderRNN(input_lang.n_words, hidden_size).to(device)
#bleu_decoder_eval = DecoderRNN(hidden_size, output_lang.n_words).to(device)
bleu_decoder_eval = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

best_bleu_translations = evaluateTranslation(bleu_encoder_eval, bleu_decoder_eval, "best_bleu_encoder.pt", "best_bleu_decoder.pt", test_pairs)

In [ ]:
# Visualizando traduções geradas
showTranslations(best_valid_translations, 20)

In [ ]:
# Visualizando traduções geradas
showTranslations(best_bleu_translations, 20)

In [ ]:
print("Modelo com melhor desempenho de valid_loss:")
valid_translations_matches = count_matches(best_valid_translations)

print()

print("Modelo com melhor desempenho de BLEU Score:")
bleu_translations_matches = count_matches(best_bleu_translations)

In [ ]:
showTranslations(bleu_translations_matches, len(bleu_translations_matches))

========== outros testes ==========

In [ ]:
# Testando o modelo configurado a partir do treinamento com melhor desempenho baseado na BLEU Score
bleu_encoder_eval = EncoderRNN(input_lang.n_words, hidden_size).to(device)
#bleu_decoder_eval = DecoderRNN(hidden_size, output_lang.n_words).to(device)
bleu_decoder_eval = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

best_bleu_translations = evaluateTranslation(bleu_encoder_eval, bleu_decoder_eval, "0.0575_attn_encoder.pt", "0.0575_attn_decoder.pt", test_pairs)

In [ ]:
# Visualizando traduções geradas
showTranslations(best_bleu_translations, 20)

In [ ]:
print("Modelo com melhor desempenho de BLEU Score:")
bleu_translations_matches = count_matches(best_bleu_translations)

In [ ]:
showTranslations(bleu_translations_matches, len(bleu_translations_matches))